# Домашнее задание №5

## Классические методы сегментации

### Цель

Сравнить классические методы сегментации из разных семейств (пороговые и региональные, кластеризационные, графовые) на одних и тех же изображениях по количественным метрикам качества и по времени работы и определить, какие свойства сцены «ломают» каждый метод.

Результатом работы является не подобранный под одну картинку набор параметров, а сопоставимое сравнение методов с указанием условий, в которых сделан вывод.

[Методические указания блока](README.md) · [Общие МУ](../../../docs/guidelines-students.md) · [Рубрика оценивания](../teachers-assessment/README.md)

## 1. Что используется в работе

Библиотеки: `opencv-python`, `numpy`, `scikit-image`, `matplotlib`, `pandas`.

Данные. Ноутбук работает без интернета и состоит из двух частей:

- **синтетический набор** с точной эталонной маской — по нему считаются IoU и Dice. Варьируются контраст объекта и фона, текстурность фона, уровень шума, размер объекта;
- **реальные изображения** из `skimage.data` (`coins`, `astronaut`) — по ним оцениваются качественные эффекты, которых нет на синтетике (неоднородное освещение, размытые границы, тени).

Для реальных изображений эталонной разметки нет. Количественная оценка на них возможна только после вашей собственной разметки или после подключения набора с готовой разметкой (карточка BSDS500 в [resources/datasets](../../resources/datasets/README.md)). Не выдавайте визуальное впечатление за метрику: оценка сегментации только «на глаз» прямо указана в [типичных ошибках](../teachers-assessment/README.md#типичные-ошибки).

Заготовка содержит: генератор синтетики с эталоном, метрики IoU/Dice, приведение мультисегментной разметки к бинарной, замер времени, журнал экспериментов, каркас сводных таблиц и один полностью реализованный метод (baseline).

In [ ]:
# Зависимости (при необходимости раскомментируйте):
# %pip install opencv-python numpy scikit-image matplotlib pandas

import platform
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage
from skimage import data, segmentation, filters

SEED = 42
rng = np.random.default_rng(SEED)
cv2.setRNGSeed(SEED)

OUTPUT_DIR = Path("outputs_hw5")
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

print("python      :", platform.python_version())
print("opencv      :", cv2.__version__)
print("numpy       :", np.__version__)
print("scikit-image:", skimage.__version__)
print("pandas      :", pd.__version__)
print("SEED        :", SEED)

## 2. Краткая теоретическая справка

### 2.1. Пороговые методы

Метод Оцу выбирает порог $t$, максимизирующий межклассовую дисперсию яркостей:

$$\sigma_b^2(t) = \omega_0(t)\,\omega_1(t)\,\bigl[\mu_0(t) - \mu_1(t)\bigr]^2 \rightarrow \max_t,$$

где $\omega_i$ — доли пикселей классов, $\mu_i$ — их средние яркости. Метод предполагает бимодальную гистограмму и глобально постоянное освещение. При градиенте освещённости глобальный порог систематически отрезает часть объекта — это и проверяется на синтетике с градиентным фоном.

### 2.2. Региональные методы

Разделение-слияние: изображение рекурсивно делится на блоки, пока внутри блока не выполнен критерий однородности (например, $\max - \min < T$ или $\sigma < T$), затем соседние однородные блоки сливаются. Метод оперирует связностью и однородностью, а не глобальной гистограммой, поэтому устойчивее к плавному изменению яркости, но чувствителен к порогу однородности и к текстуре.

### 2.3. Кластеризационные методы

k-means минимизирует суммарную внутрикластерную дисперсию признаков:

$$J = \sum_{i=1}^{k} \sum_{\mathbf{f} \in C_i} \lVert \mathbf{f} - \boldsymbol{\mu}_i \rVert^2 .$$

Признак пикселя может включать координаты:

$$\mathbf{f}(x,y) = \bigl(L, a, b, \lambda x, \lambda y\bigr),$$

где $\lambda$ задаёт вес пространственной близости относительно цветовой. Число кластеров $k$ задаётся заранее.

Сдвиг среднего значения (mean shift) ищет моды плотности признаков, итеративно смещая точку на вектор

$$\mathbf{m}(\mathbf{x}) = \frac{\sum_i K(\mathbf{x}_i - \mathbf{x})\,\mathbf{x}_i}{\sum_i K(\mathbf{x}_i - \mathbf{x})} - \mathbf{x}.$$

Число сегментов не задаётся: оно определяется числом найденных мод, а значит, косвенно — шириной ядра.

### 2.4. Графовые методы

Изображение представляется взвешенным графом $G=(V,E)$, вес ребра выражает сходство пикселей. Нормализованный разрез:

$$\mathrm{Ncut}(A,B) = \frac{\mathrm{cut}(A,B)}{\mathrm{assoc}(A,V)} + \frac{\mathrm{cut}(A,B)}{\mathrm{assoc}(B,V)},$$

где $\mathrm{cut}(A,B) = \sum_{u \in A, v \in B} w(u,v)$, $\mathrm{assoc}(A,V) = \sum_{u \in A, t \in V} w(u,t)$. Нормировка на $\mathrm{assoc}$ штрафует отрезание маленьких изолированных областей, к чему склонен обычный минимальный разрез. Алгоритм Фельзенсвальба (Felzenszwalb) решает близкую задачу жадно, сравнивая внутреннюю вариацию компонент с весом соединяющего ребра.

### 2.5. Метрики

$$\mathrm{IoU} = \frac{|P \cap G|}{|P \cup G|}, \qquad \mathrm{Dice} = \frac{2\,|P \cap G|}{|P| + |G|},$$

где $P$ — предсказанная область объекта, $G$ — эталонная. Обе метрики монотонно связаны, но Dice менее чувствителен к небольшим объектам. Сравнение времени работы имеет смысл только при одинаковом разрешении входа (см. [типичные ошибки](../teachers-assessment/README.md#типичные-ошибки)).

## 3. Задачи

Формулировка из [методических указаний блока](README.md#дз5-методы-сегментации):

Реализуйте/примените и сравните не менее трёх классических методов сегментации: пороговые и региональные (разделение-слияние), кластеризационные (k-means по цвету и координатам, сдвиг среднего значения — mean shift), графовые (нормализованный разрез или Felzenszwalb). Оцените результаты на изображениях с эталонной разметкой по IoU/Dice.

**Результат:** сравнительная таблица методов по качеству и времени; анализ, какие сцены «ломают» каждый метод.

Проверяемые элементы ([рубрика](../teachers-assessment/README.md)): не менее трёх методов из разных семейств; IoU/Dice посчитаны по эталонной разметке; сравнение по качеству и времени на одних и тех же изображениях.

## 4. Данные

Синтетический набор строится так, чтобы каждый фактор менялся осознанно: яркостный контраст, текстура фона, шум, градиент освещённости, размер объекта, цветовое различие. Значения подобраны так, чтобы разрыв яркостей был сопоставим с амплитудой мешающего фактора: набор, который решается глобальным порогом целиком, ничего не показывает.

Отдельно стоит случай `color_only`: яркости объекта и фона почти совпадают, различие есть только в цвете. Методы, работающие по одноканальному изображению, на нём отказывают в принципе.

Эталонная маска здесь точна по построению, поэтому расхождение метрик отражает поведение метода, а не качество разметки.

In [ ]:
def make_synthetic_segmentation_dataset(size=(240, 320), seed=SEED):
    """Синтетический набор изображений с точной эталонной маской.

    Выход: список словарей с ключами
        'name'  — идентификатор изображения;
        'image' — (H, W, 3) uint8, BGR;
        'mask'  — (H, W) uint8 из {0, 1}, 1 — объект;
        'meta'  — словарь варьируемых факторов.
    """
    h, w = size
    generator = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:h, 0:w]

    # Факторы подобраны так, чтобы ни один метод не решал все случаи:
    # gap = |fg - bg| сопоставим с амплитудой шума, текстуры или градиента.
    configs = [
        dict(name="high_contrast",  fg=210, bg=60,  noise=5,  texture=0.0,  gradient=0,   scale=1.00, color=15),
        dict(name="low_contrast",   fg=120, bg=100, noise=12, texture=0.0,  gradient=0,   scale=1.00, color=15),
        dict(name="noisy",          fg=165, bg=105, noise=35, texture=0.0,  gradient=0,   scale=1.00, color=15),
        dict(name="textured_bg",    fg=175, bg=95,  noise=5,  texture=70.0, gradient=0,   scale=1.00, color=15),
        dict(name="illum_gradient", fg=170, bg=90,  noise=5,  texture=0.0,  gradient=150, scale=1.00, color=15),
        dict(name="small_object",   fg=205, bg=65,  noise=8,  texture=0.0,  gradient=0,   scale=0.30, color=15),
        dict(name="color_only",     fg=130, bg=128, noise=5,  texture=0.0,  gradient=0,   scale=1.00, color=75),
    ]

    dataset = []
    for cfg in configs:
        mask = np.zeros((h, w), dtype=np.uint8)
        s = cfg["scale"]
        cv2.ellipse(mask, (int(0.38 * w), int(0.45 * h)),
                    (int(0.18 * w * s), int(0.26 * h * s)), 20, 0, 360, 1, -1)
        polygon = np.array([[0.62 * w, 0.25 * h], [0.86 * w, 0.38 * h],
                            [0.78 * w, 0.72 * h], [0.58 * w, 0.60 * h]], dtype=np.float32)
        centroid = polygon.mean(axis=0)
        polygon = (centroid + (polygon - centroid) * s).astype(np.int32)
        cv2.fillPoly(mask, [polygon], 1)

        image = np.where(mask == 1, float(cfg["fg"]), float(cfg["bg"]))
        if cfg["texture"] > 0:
            texture = np.sin(xx / 6.0) * np.cos(yy / 9.0)
            image = image + cfg["texture"] * texture * (mask == 0)
        if cfg["gradient"] > 0:
            image = image + cfg["gradient"] * (xx / float(w))
        image = image + generator.normal(0.0, cfg["noise"], (h, w))
        image = np.clip(image, 0, 255).astype(np.uint8)

        # Цветовое различие вводится с компенсацией яркости: сдвиг канала R
        # гасится сдвигом G так, что серое изображение не меняется. Поэтому
        # цветовой и яркостный факторы разделены по построению.
        color = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR).astype(np.int32)
        color[..., 2] += cfg["color"] * mask
        color[..., 1] -= int(round(0.299 / 0.587 * cfg["color"])) * mask
        color = np.clip(color, 0, 255).astype(np.uint8)
        dataset.append({"name": cfg["name"], "image": color, "mask": mask, "meta": cfg})
    return dataset


synthetic = make_synthetic_segmentation_dataset()

fig, axes = plt.subplots(2, len(synthetic), figsize=(3 * len(synthetic), 6))
for col, item in enumerate(synthetic):
    axes[0, col].imshow(cv2.cvtColor(item["image"], cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(item["name"], fontsize=9)
    axes[1, col].imshow(item["mask"], cmap="gray")
    axes[1, col].set_title("эталон", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def load_real_images():
    """Реальные изображения без эталонной разметки (качественный анализ).

    Выход: список словарей 'name' / 'image' (BGR uint8) / 'mask' (None).
    """
    coins = cv2.cvtColor(data.coins(), cv2.COLOR_GRAY2BGR)
    astronaut = cv2.cvtColor(data.astronaut(), cv2.COLOR_RGB2BGR)
    return [
        {"name": "coins", "image": coins, "mask": None, "meta": {"source": "skimage.data.coins"}},
        {"name": "astronaut", "image": astronaut, "mask": None,
         "meta": {"source": "skimage.data.astronaut"}},
    ]


real_images = load_real_images()

# TODO (по желанию): подключите изображения с эталонной разметкой (BSDS500 или
# собственная разметка). Требование к формату: mask — (H, W) uint8 из {0, 1},
# размер совпадает с изображением. Тогда эти изображения войдут в общие метрики.

fig, axes = plt.subplots(1, len(real_images), figsize=(10, 4))
for ax, item in zip(np.atleast_1d(axes), real_images):
    ax.imshow(cv2.cvtColor(item["image"], cv2.COLOR_BGR2RGB))
    ax.set_title(item["name"])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Метрики, приведение разметки и журнал

Большинство методов из задания дают **мультисегментную** разметку (k-means, mean shift, Felzenszwalb), а эталон бинарный. Прямое сравнение невозможно, поэтому нужна процедура приведения.

Функция `labels_to_binary_oracle` выбирает подмножество сегментов, максимизирующее IoU с эталоном. Это **оптимистичная (oracle) оценка**: она использует эталон и потому даёт верхнюю границу качества метода при идеальном отборе сегментов. Так сравнивать методы допустимо, но в отчёте это ограничение необходимо назвать явно и, желательно, дополнить честной процедурой отбора без эталона (например, сегмент, содержащий заданную точку, или сегмент с максимальной средней яркостью).

In [ ]:
def iou_score(gt, pred):
    """IoU для бинарных масок. Вход: (H, W) {0,1} или bool. Выход: float."""
    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)
    union = np.logical_or(gt, pred).sum()
    return 1.0 if union == 0 else float(np.logical_and(gt, pred).sum() / union)


def dice_score(gt, pred):
    """Коэффициент Дайса для бинарных масок."""
    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)
    denom = gt.sum() + pred.sum()
    return 1.0 if denom == 0 else float(2.0 * np.logical_and(gt, pred).sum() / denom)


def labels_to_binary_oracle(labels, gt_mask):
    """Привести мультисегментную разметку к бинарной по эталону (oracle).

    Вход:  labels (H, W) int — номера сегментов; gt_mask (H, W) {0,1}.
    Выход: (binary_mask (H, W) uint8, chosen_labels list).

    Каждый сегмент относится к объекту, если внутри него доля пикселей эталона
    больше 0.5. Оценка оптимистична: используется эталон.
    """
    labels = np.asarray(labels)
    gt = np.asarray(gt_mask).astype(bool)
    binary = np.zeros(labels.shape, dtype=np.uint8)
    chosen = []
    for value in np.unique(labels):
        region = labels == value
        if region.sum() == 0:
            continue
        if gt[region].mean() > 0.5:
            binary[region] = 1
            chosen.append(int(value))
    return binary, chosen


RUNS = []


def log_run(**fields):
    """Добавить запись в журнал экспериментов (конфигурация + метрики + время)."""
    record = {"run_id": len(RUNS), **fields}
    RUNS.append(record)
    return record


def runs_table(columns=None, sort_by=None):
    if not RUNS:
        return pd.DataFrame()
    frame = pd.DataFrame(RUNS)
    if columns:
        frame = frame[[c for c in columns if c in frame.columns]]
    if sort_by:
        frame = frame.sort_values(sort_by)
    return frame.reset_index(drop=True)


def save_runs(path=OUTPUT_DIR / "runs.csv"):
    runs_table().to_csv(path, index=False)
    return path

## 6. Единый интерфейс метода и опорный запуск

Все методы приводятся к одному интерфейсу: изображение BGR на входе, целочисленная разметка сегментов на выходе. Это позволяет прогонять их одним и тем же кодом и получать сопоставимые время и метрики.

Ниже полностью реализован пороговый baseline (метод Оцу). Это опорная точка серии: она не меняется после начала эксперимента.

In [ ]:
def segment_otsu(image_bgr, blur_ksize=5):
    """Опорный метод: глобальный порог Оцу по яркости.

    Вход:  image_bgr (H, W, 3) uint8.
    Выход: labels (H, W) int32 со значениями {0, 1}.
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    if blur_ksize and blur_ksize > 1:
        gray = cv2.GaussianBlur(gray, (blur_ksize, blur_ksize), 0)
    _, binary = cv2.threshold(gray, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary.astype(np.int32)


def run_method(method_fn, dataset, method_name, params=None, resize_to=None):
    """Прогнать метод по набору изображений с эталоном и записать журнал.

    Вход:
        method_fn  — функция image_bgr -> labels (H, W) int;
        dataset    — список словарей 'name'/'image'/'mask';
        method_name— имя метода для таблиц;
        params     — словарь параметров (попадает в журнал одной строкой);
        resize_to  — (w, h) или None; одинаковое разрешение обязательно,
                     иначе сравнение времён некорректно.
    Выход: (results DataFrame, masks dict name -> предсказанная бинарная маска).
    """
    params = params or {}
    masks = {}
    rows = []
    for item in dataset:
        image = item["image"]
        gt = item["mask"]
        if resize_to is not None:
            image = cv2.resize(image, resize_to, interpolation=cv2.INTER_AREA)
            if gt is not None:
                gt = cv2.resize(gt, resize_to, interpolation=cv2.INTER_NEAREST)
        start = time.perf_counter()
        labels = method_fn(image)
        elapsed = time.perf_counter() - start

        if gt is None:
            iou = dice = float("nan")
            binary = (np.asarray(labels) > 0).astype(np.uint8)
        else:
            binary, _ = labels_to_binary_oracle(labels, gt)
            iou = iou_score(gt, binary)
            dice = dice_score(gt, binary)

        masks[item["name"]] = binary
        rows.append(log_run(method=method_name, params=str(params), image=item["name"],
                            n_segments=int(len(np.unique(labels))),
                            resolution=f"{image.shape[1]}x{image.shape[0]}",
                            iou=round(iou, 4), dice=round(dice, 4),
                            seconds=round(elapsed, 4), seed=SEED))
    return pd.DataFrame(rows), masks


def show_masks(dataset, masks, title=""):
    """Контактный лист: изображение / эталон / предсказание."""
    n = len(dataset)
    fig, axes = plt.subplots(3, n, figsize=(3 * n, 8))
    for col, item in enumerate(dataset):
        axes[0, col].imshow(cv2.cvtColor(item["image"], cv2.COLOR_BGR2RGB))
        axes[0, col].set_title(item["name"], fontsize=9)
        axes[1, col].imshow(item["mask"] if item["mask"] is not None
                            else np.zeros(item["image"].shape[:2]), cmap="gray")
        axes[1, col].set_title("эталон", fontsize=9)
        axes[2, col].imshow(masks[item["name"]], cmap="gray")
        axes[2, col].set_title("предсказание", fontsize=9)
    for ax in axes.ravel():
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Рельсы: baseline на синтетическом наборе при фиксированном разрешении.
RESIZE_TO = (320, 240)  # (w, h) — одинаково для всех методов

baseline_results, baseline_masks = run_method(
    segment_otsu, synthetic, method_name="otsu", params={"blur_ksize": 5},
    resize_to=RESIZE_TO,
)
display(baseline_results[["image", "iou", "dice", "seconds", "n_segments"]].round(3))
print("Средний IoU baseline:", round(baseline_results["iou"].mean(), 3))

show_masks(synthetic, baseline_masks, "Метод Оцу (baseline)")

## 7. Методы из остальных семейств

Реализуйте не менее трёх методов дополнительно к baseline, покрыв разные семейства: региональное (разделение-слияние), кластеризационное (k-means и/или mean shift), графовое (Felzenszwalb или нормализованный разрез).

Инструменты, доступные без интернета:

- k-means: `cv2.kmeans` по признаку $(L, a, b, \lambda x, \lambda y)$ (`cv2.cvtColor(..., cv2.COLOR_BGR2LAB)`);
- mean shift: `cv2.pyrMeanShiftFiltering` с последующей маркировкой связных областей (`cv2.connectedComponents` по квантованному результату);
- графовый: `skimage.segmentation.felzenszwalb`; для нормализованного разреза — `skimage.graph.rag_mean_color` + `skimage.graph.cut_normalized` (в старых версиях `skimage.future.graph`), с обёрткой в `try/except ImportError`;
- разделение-слияние: собственная рекурсия по квадрантам с критерием однородности и последующим слиянием соседей.

Важно: пространственный вес $\lambda$ в k-means и ширина ядра в mean shift — не «настройки по вкусу», а исследуемые факторы. Меняйте по одному.

In [ ]:
def segment_image(image_bgr, method, **params):
    """Единый интерфейс метода сегментации.

    Контракт.
    Вход:
        image_bgr — (H, W, 3) uint8, BGR;
        method    — 'region' | 'kmeans' | 'meanshift' | 'felzenszwalb' | 'ncut';
        params    — параметры метода (k, spatial_weight, sp, sr, scale, min_size,
                    homogeneity_threshold и т. п.).
    Выход:
        labels — (H, W) int32, номера сегментов начиная с 0; число сегментов
                 произвольно (для бинарных методов — {0, 1}).

    Требования:
        1) функция не должна использовать эталонную маску;
        2) все параметры передаются явно и логируются;
        3) вход не масштабируется внутри функции — разрешение задаётся снаружи,
           иначе сравнение времён теряет смысл.
    """
    raise NotImplementedError("TODO (задание 3): реализуйте методы сегментации")


# TODO (задание 3): реализуйте методы и прогоните их через run_method.
#
# from functools import partial
# METHODS = {
#     "kmeans_color":   partial(segment_image, method="kmeans", k=3, spatial_weight=0.0),
#     "kmeans_spatial": partial(segment_image, method="kmeans", k=3, spatial_weight=0.5),
#     "meanshift":      partial(segment_image, method="meanshift", sp=12, sr=24),
#     "felzenszwalb":   partial(segment_image, method="felzenszwalb", scale=200, min_size=80),
#     "region_merge":   partial(segment_image, method="region", homogeneity_threshold=18),
# }
# for name, fn in METHODS.items():
#     results, masks = run_method(fn, synthetic, method_name=name,
#                                 params={...}, resize_to=RESIZE_TO)
#     show_masks(synthetic, masks, name)

runs_table()

In [ ]:
# TODO (задание 3): контролируемая серия по одному фактору.
#
# Минимум одна серия на метод, например:
#   k-means:      k in [2, 3, 4, 6] при фиксированном spatial_weight;
#   k-means:      spatial_weight in [0, 0.25, 0.5, 1.0] при фиксированном k;
#   mean shift:   sr in [8, 16, 24, 32] при фиксированном sp;
#   felzenszwalb: scale in [50, 100, 200, 400] при фиксированном min_size.
#
# Для каждой конфигурации фиксируйте: число сегментов, IoU, Dice, время.
# Отдельно отметьте, как число сегментов связано с качеством после приведения
# к бинарной маске: рост числа сегментов может улучшать oracle-IoU, не улучшая
# практическую применимость метода.

pass

In [ ]:
# TODO (задание 3): качественная проверка на реальных изображениях.
#
# Прогоните те же методы на real_images (эталона нет, метрики будут NaN) и
# зафиксируйте наблюдения: где методы дают осмысленные границы, где рассыпаются.
# Если вы разметили изображения вручную, добавьте маски в поле 'mask' и
# включите эти изображения в количественное сравнение.
#
# results, masks = run_method(METHODS["felzenszwalb"], real_images,
#                             method_name="felzenszwalb", params={...},
#                             resize_to=RESIZE_TO)

pass

## Отчёт

### Сводные таблицы

Обязательны:

1. «метод × изображение → IoU» (одно и то же разрешение для всех методов);
2. «метод → средний IoU, средний Dice, среднее и p95 времени, среднее число сегментов»;
3. отдельная таблица серии по исследуемому фактору для каждого метода.

In [ ]:
frame = runs_table()

if frame.empty:
    print("Журнал пуст: выполните разделы 6 и 7.")
else:
    per_image = frame.pivot_table(index="method", columns="image", values="iou", aggfunc="mean")
    display(per_image.round(3))

    summary = frame.groupby("method").agg(
        iou_mean=("iou", "mean"),
        iou_min=("iou", "min"),
        dice_mean=("dice", "mean"),
        seconds_mean=("seconds", "mean"),
        seconds_p95=("seconds", lambda s: float(np.percentile(s, 95))),
        segments_mean=("n_segments", "mean"),
    )
    display(summary.round(3))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(summary["seconds_mean"], summary["iou_mean"])
    for name, row in summary.iterrows():
        ax.annotate(name, (row["seconds_mean"], row["iou_mean"]),
                    textcoords="offset points", xytext=(5, 4), fontsize=9)
    ax.set_xlabel("среднее время, с")
    ax.set_ylabel("средний IoU")
    ax.set_title("качество против стоимости (одно разрешение входа)")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

save_runs()

### Выводы

**Наблюдения** (только измеренные факты со ссылкой на строки журнала):

-

**Интерпретация** (какое свойство сцены объясняет отказ каждого метода: низкий контраст, текстура фона, градиент освещённости, размер объекта, шум):

-

**Выводы и их границы** (на каких изображениях, при каком разрешении, при каких диапазонах параметров получен результат; отдельно укажите, что оценка по `labels_to_binary_oracle` оптимистична):

-

**Анализ ошибок**: приведите не менее трёх примеров с визуализацией — по одному характерному отказу на семейство методов.

## Контрольные вопросы

Из [списка вопросов блока](README.md#контрольные-вопросы-блока):

4. Почему mean shift не требует задания числа сегментов, а k-means требует?
5. В чём идея нормализованного разреза графа?

Дополнительно:

- Чем IoU отличается от Dice и в каких случаях они расходятся сильнее всего?
- Что даст переразметка эталона другим человеком для ваших метрик сегментации (вопрос к защите)?

## Чек-лист перед сдачей

- [ ] Ноутбук исполняется сверху вниз без ошибок после `Restart & Run All`.
- [ ] Указаны ФИО, группа, номер работы, источники данных.
- [ ] Seed и версии библиотек зафиксированы и выведены.
- [ ] Сравнены не менее трёх методов из разных семейств плюс baseline.
- [ ] Все методы прогнаны на одних и тех же изображениях и при одинаковом разрешении входа.
- [ ] IoU и Dice посчитаны по эталонной разметке, а не описаны словесно.
- [ ] Для каждого метода проведена серия по одному фактору.
- [ ] Оптимистичность oracle-приведения мультисегментной разметки к бинарной оговорена в отчёте.
- [ ] Есть сводные таблицы, график «качество против времени», журнал сохранён (`outputs_hw5/runs.csv`).
- [ ] Разобраны характерные отказы каждого метода с визуализацией.
- [ ] Наблюдения, интерпретация и выводы разделены; указаны ограничения.